# 🎹 Neural Swipe Training

**Before running:** Go to `Runtime → Change runtime type → GPU (T4)`

This notebook will:
1. Clone the neural-swipe-typing repo
2. Upload your training data
3. Train the model (~2-3 hours)
4. Save the model to Google Drive

## Step 1: Check GPU

In [ ]:
# Verify GPU is available
!nvidia-smi
import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

## Step 2: Clone Repository & Install Dependencies

In [ ]:
# Clone the neural-swipe-typing repo
!git clone https://github.com/proshian/neural-swipe-typing.git
%cd neural-swipe-typing

# Install dependencies
!pip install -q pytorch-lightning tqdm

## Step 3: Upload Your Training Data

Run this cell, then **upload the files from your laptop**:
- `train.jsonl`
- `valid.jsonl`
- `voc.txt`
- `gridname_to_grid.json`
- `trajectory_features_statistics.json`
- `key_bounding_boxes.json`
- `en.json` (keyboard tokenizer)
- `train_english.json` (config)

In [ ]:
from google.colab import files
import os

# Create directories
os.makedirs('data/english', exist_ok=True)
os.makedirs('tokenizers/keyboard', exist_ok=True)
os.makedirs('configs/train', exist_ok=True)

print("📁 Upload your training files when prompted...")
print("You need to upload 8 files total.\n")

uploaded = files.upload()

In [ ]:
# Move uploaded files to correct locations
import shutil

file_destinations = {
    'train.jsonl': 'data/english/train.jsonl',
    'valid.jsonl': 'data/english/valid.jsonl',
    'voc.txt': 'data/english/voc.txt',
    'gridname_to_grid.json': 'data/english/gridname_to_grid.json',
    'trajectory_features_statistics.json': 'data/english/trajectory_features_statistics.json',
    'key_bounding_boxes.json': 'data/english/key_bounding_boxes.json',
    'en.json': 'tokenizers/keyboard/en.json',
    'train_english.json': 'configs/train/train_english.json',
}

for filename, dest in file_destinations.items():
    if os.path.exists(filename):
        shutil.move(filename, dest)
        print(f"✅ Moved {filename} → {dest}")
    else:
        print(f"⚠️ Missing: {filename}")

## Step 4: Fix Config Paths for Colab

In [ ]:
import json

# Load and fix paths in config
with open('configs/train/train_english.json', 'r') as f:
    config = json.load(f)

# Update paths to be relative to Colab working directory
config['grids_path'] = './data/english/gridname_to_grid.json'
config['trajectory_features_statistics_path'] = './data/english/trajectory_features_statistics.json'
config['bounding_boxes_path'] = './data/english/key_bounding_boxes.json'
config['keyboard_tokenizer_path'] = './tokenizers/keyboard/en.json'
config['vocab_path'] = './data/english/voc.txt'
config['dataset_paths'] = {
    'train': './data/english/train.jsonl',
    'val': './data/english/valid.jsonl'
}

# Use GPU and larger batch size (we have VRAM now!)
config['device'] = 'cuda'
config['train_batch_size'] = 256
config['val_batch_size'] = 256
config['dataloader_num_workers'] = 2

with open('configs/train/train_english.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✅ Config updated for Colab")
print(f"   Batch size: {config['train_batch_size']}")
print(f"   Device: {config['device']}")

## Step 5: Patch train.py for GPU

In [ ]:
# Patch train.py to use config device instead of hardcoded 'gpu'
with open('src/train.py', 'r') as f:
    train_code = f.read()

train_code = train_code.replace(
    "accelerator = 'gpu'",
    'accelerator = train_config.get("device", "gpu")'
)

with open('src/train.py', 'w') as f:
    f.write(train_code)

print("✅ Patched train.py to use config device")

## Step 6: Start Training! 🚀

This will take **2-3 hours**. You can close the tab and come back—just don't let it sit idle too long or Colab will disconnect.

In [ ]:
# Start training
!python -m src.train --train_config configs/train/train_english.json

## Step 7: Save Model to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create folder and copy checkpoints
import shutil
import os

dest = '/content/drive/MyDrive/neural_swipe_model'
os.makedirs(dest, exist_ok=True)

# Copy the best checkpoint
if os.path.exists('checkpoints'):
    shutil.copytree('checkpoints', f'{dest}/checkpoints', dirs_exist_ok=True)
    print(f"✅ Saved checkpoints to Google Drive: {dest}")
else:
    print("⚠️ No checkpoints found")

## Done! 🎉

Your trained model is saved to Google Drive under `neural_swipe_model/checkpoints/`.

Download the `.ckpt` file and we'll convert it to ExecuTorch format for your Android keyboard.